In [1]:
# Install required dependencies
# The '!' tells Jupyter to run this as a terminal command
!pip install -qU langchain-groq langchain-community langchain-classic wikipedia numexpr

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
import os
from langchain_groq import ChatGroq
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import Tool
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

# 1. API Key Setup (Replace with your actual key)
GROQ_API_KEY = "YOUR_API_KEY"

# 2. The Brain
llm = ChatGroq(
    temperature=0,
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile"
)

# 3. The Hands (Custom formatted tools)
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1500)
base_wiki = WikipediaQueryRun(api_wrapper=api_wrapper)

def clean_wiki(query):
    result = base_wiki.run(query)
    return f"\n{result}...\n\n"

wiki_tool = Tool(
    name="wikipedia",
    func=clean_wiki,
    description="Searches Wikipedia for current events, facts, or people."
)

base_math = load_tools(["llm-math"], llm=llm)[0]

def clean_math(query):
    result = base_math.run(query)
    return f"\n{result}\n\n"

math_tool = Tool(
    name="Calculator",
    func=clean_math,
    description="Useful for when you need to answer questions about math."
)

tools = [wiki_tool, math_tool]

# 4. The Persona
template = '''Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the exact following format and ALWAYS include blank lines between steps:

Question: the input question you must answer

Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action

... (this Thought/Action/Action Input/Observation can repeat N times)

Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}'''

prompt = PromptTemplate.from_template(template)

# 5. The Orchestrator & Executor
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=4,
    early_stopping_method="generate",
    handle_parsing_errors=True
)

print("✅ Agent successfully initialized and ready!")

✅ Agent successfully initialized and ready!


In [3]:
question_1 = "What is the square root of 74849 multiplied by 13.5?"

print("--- RAW CHATBOT MODE (NO TOOLS) ---")
# This bypasses the agent and asks the raw language model
raw_response = llm.invoke(question_1)
print(f"Answer: {raw_response.content}\n")

print("--- AGENT MODE (TOOLS ACTIVE) ---")
# This routes the question through the LangChain agent
agent_executor.invoke({"input": question_1})

--- RAW CHATBOT MODE (NO TOOLS) ---
Answer: To find the result, first, we need to calculate the square root of 74849. 

The square root of 74849 is 273.97 (rounded to two decimal places).

Now, we multiply 273.97 by 13.5.

273.97 * 13.5 = 3703.645

So, the result is approximately 3703.65.

--- AGENT MODE (TOOLS ACTIVE) ---


> Entering new AgentExecutor chain...
To find the square root of 74849 multiplied by 13.5, I first need to calculate the square root of 74849.

Action: Calculator
Action Input: sqrt(74849)
Answer: 273.5854528296415

Now that I have the square root of 74849, I need to multiply it by 13.5 to get the final result.

Action: Calculator
Action Input: 273.5854528296415 * 13.5
Answer: 3693.40361320016

I now know the final answer
Final Answer: 3693.40361320016

> Finished chain.


{'input': 'What is the square root of 74849 multiplied by 13.5?',
 'output': '3693.40361320016'}

In [4]:
question_2 = "What is the age of Eric Adams in 2026 multiplied by 5?"

print("--- AGENT MODE (TOOLS ACTIVE) ---")
agent_executor.invoke({"input": question_2})

--- AGENT MODE (TOOLS ACTIVE) ---


> Entering new AgentExecutor chain...
To find the age of Eric Adams in 2026 multiplied by 5, I first need to find out how old Eric Adams will be in 2026.

Action: wikipedia
Action Input: Eric Adams
Page: Eric Adams
Summary: Eric Leroy Adams (born September 1, 1960) is an American politician and former police officer who served as the 111th mayor of New York City from 2022 to 2025. A member of the Democratic Party, Adams was an officer in the New York City Transit Police and then the New York City Police Department (NYPD) for more than 20 years, retiring at the rank of captain. He served in the New York State Senate from 2006 to 2013, representing the 20th district in Brooklyn. In 2013, Adams became the first black American to be elected Brooklyn Borough President; he was re-elected in 2017.
In 2021, Adams received the Democratic Party's nomination for mayor of New York City after winning a crowded primary that used ranked-choice voting. In the genera

{'input': 'What is the age of Eric Adams in 2026 multiplied by 5?',
 'output': '330'}